# Aria HLOC: indoor LaMAria (Gen 1)

Build two HLOC maps from **`R_03_easy`**: a native SLAM-camera map from both Gen 1 tracking cameras and an RGB map. Then localize non-keyframes from the same recording with the matching query type. This indoor office/lab example uses the original public VRS and genuine MPS trajectory/online calibration. It is **not an independent-recording accuracy benchmark**.

First follow [the README installation](README.md#install-the-pinned-environment) and [LaMAria download/MPS guide](docs/lamaria.md). This notebook does not download data, accept terms or start an MPS request. Model weights download on first HLOC inference. Source cells intentionally have no executed outputs.

For Gen 2 separate-recording queries and recorded VIO, use the preserved [Pilot notebook](pilot_replay.ipynb). R_03 has no recorded VIO; its closed-loop MPS must not be presented as VIO.

In [ ]:
import json
from pathlib import Path
import torch  # Load torch before COLMAP; see the README macOS OpenMP note.
from build_map import build_map
from check_inputs import check_inputs
from map_utils import file_sha256, Trajectory, write_json
from query_map import load_map, query_map

DATA = Path('data/lamaria')
VRS = DATA / 'R_03_easy.vrs'
MPS = DATA / 'mps_R_03_easy_vrs'
SLAM_MAP = Path('maps/lamaria_R03_slam')
RGB_MAP = Path('maps/lamaria_R03_rgb')
SLAM_CACHE = Path('queries/lamaria_R03_slam_map_slam.json')
RGB_CACHE = Path('queries/lamaria_R03_rgb_map_rgb.json')
SOURCE_SHA256 = '52962a4934d2b94d77ec276d7cf5579065e598c8e8448e2fd9d87e2bbfdb6229'
required = (VRS, MPS / 'slam/closed_loop_trajectory.csv', MPS / 'slam/online_calibration.jsonl')
if any(not path.is_file() for path in required):
    raise FileNotFoundError('Obtain the original R_03_easy VRS and its real MPS outputs using docs/lamaria.md')
assert VRS.stat().st_size == 1135832862
assert file_sha256(VRS) == SOURCE_SHA256
trajectory = Trajectory(MPS / 'slam/closed_loop_trajectory.csv')
assert trajectory.graph_uid
print(json.dumps(check_inputs(VRS, MPS, camera_set='all'), indent=2))

## 1. Build the native SLAM-camera map and the RGB map

Known MPS poses triangulate motion-separated keyframes from each camera set: `camera_set='slam'` (the builder default) uses both native Gen 1 tracking cameras, and `camera_set='rgb'` uses the RGB camera. `max_keyframes=0` retains all selected frames; calibration and signed timestamps come from this recording. Run once with fresh output directories. If the exact maps already exist, **skip this cell** and verify their fingerprints below; do not automatically rebuild or overwrite them.

In [ ]:
for map_path, camera_set in ((SLAM_MAP, 'slam'), (RGB_MAP, 'rgb')):
    build_stats = build_map([VRS], [MPS], map_path, camera_set=camera_set, max_keyframes=0)
    print(camera_set, json.dumps(build_stats, indent=2))

## 2. Verify each map's source, cameras and fingerprint

Each map must contain only this original VRS fingerprint in the same MPS graph, with the expected camera set and labels. Both maps share the MPS frame and `graph_uid`, so the graph alone cannot tell them apart: keep the printed SHA-256 of each `map.json` and `sfm/points3D.bin` with its results. Legacy path-only maps cannot establish recording identity and are rejected here. This is a single mapping recording, not a claim of query-recording independence.

In [ ]:
expected_labels = {'slam': {'camera-slam-left', 'camera-slam-right'}, 'rgb': {'camera-rgb'}}
fingerprints = {}
for map_path, camera_set in ((SLAM_MAP, 'slam'), (RGB_MAP, 'rgb')):
    metadata = load_map(map_path)
    assert metadata.get('source_vrs_sha256') == [SOURCE_SHA256]
    assert metadata['camera_set'] == camera_set
    assert metadata['graph_uid'] == trajectory.graph_uid
    assert {view['camera']['label'] for view in metadata['views']} == expected_labels[camera_set]
    fingerprints[camera_set] = {name: file_sha256(map_path / name) for name in ('map.json', 'sfm/points3D.bin')}
    print(camera_set, 'map views:', len(metadata['views']), json.dumps(fingerprints[camera_set]))

## 3. Native two-camera queries on the SLAM map

Query the native `camera-slam-left`/`camera-slam-right` streams with gPnP. The adapter requires the complete generation-specific rig; it selects two Gen 1 or four Gen 2 cameras, not a partial available subset. Corrected camera times must fit the original 2 ms synchronization gate. Exact mapping keyframes are skipped using source SHA-256 plus camera/frame identity; report the actual query count, not the requested limit.

MPS query poses are used only for same-graph diagnostics, never retrieval or a pose prior. This is still the same recording and same MPS solution, not a blind or physical-ground-truth benchmark. Gate acceptance does not prove correctness.

In [ ]:
if SLAM_CACHE.exists():
    raise FileExistsError('Choose a fresh query cache path instead of overwriting evidence')
slam_result = query_map(SLAM_MAP, VRS, MPS, mode='slam', num_queries=50, stride=64)
assert slam_result['source_identity_check'] == 'sha256'
assert slam_result['query_in_mapping_recordings'] is True
assert slam_result['query_vrs_sha256'] == SOURCE_SHA256
assert slam_result['same_mps_graph'] is True
print(json.dumps({key: value for key, value in slam_result.items() if key != 'queries'}, indent=2))
SLAM_CACHE.parent.mkdir(parents=True, exist_ok=True)
write_json(SLAM_CACHE, {**slam_result, 'map_path': str(SLAM_MAP), 'map_fingerprint': fingerprints['slam']})

## 4. RGB queries on the RGB map

Stride 31 covers the whole short recording, and a high query limit lets all sampled anchors be considered. In the dated RGB-only run this retained **17 queries**, not the requested limit of 1,000. Keep this run's actual count and all failures. Either query type can also run against the other map; the recorded comparison of all four combinations is in [benchmarks/lamaria_r03_dual_map.json](benchmarks/lamaria_r03_dual_map.json).

In [ ]:
if RGB_CACHE.exists():
    raise FileExistsError('Choose a fresh query cache path instead of overwriting evidence')
rgb_result = query_map(RGB_MAP, VRS, MPS, mode='rgb', num_queries=1000, stride=31)
assert rgb_result['source_identity_check'] == 'sha256'
assert rgb_result['query_in_mapping_recordings'] is True
assert rgb_result['query_vrs_sha256'] == SOURCE_SHA256
assert rgb_result['same_mps_graph'] is True
print(json.dumps({key: value for key, value in rgb_result.items() if key != 'queries'}, indent=2))
RGB_CACHE.parent.mkdir(parents=True, exist_ok=True)
write_json(RGB_CACHE, {**rgb_result, 'map_path': str(RGB_MAP), 'map_fingerprint': fingerprints['rgb']})

## Recorded results and scope

The separately executed real-MPS runs built a native SLAM-camera map with **2,232 views (1,116 per camera) and 117,790 points** (mean native reprojection 1.348 px) and an RGB map with **930 views and 95,253 points** (1.390 px). On the same 50 matched anchors, two-camera queries placed all 48 reference-covered queries within 10 cm and 5° on both maps (warm median 2.38 s on the SLAM map, 1.98 s on the RGB map), and RGB queries did so on the RGB map (1.17 s), while the SLAM map accepted 49 of 50 RGB queries and 47 of 48 covered ones (1.70 s). All four map/query combinations used the same 50 anchors, chosen before any query with a 20 ms exclusion around every keyframe of both maps; the stride cells above do not reproduce that set. Exact counts, coverage, timings and map fingerprints are in [benchmarks/lamaria_r03_dual_map.json](benchmarks/lamaria_r03_dual_map.json).

An earlier RGB-only run (930 keyframes, 95,203 landmarks; 17 RGB and 50 two-camera queries, both against that RGB map) is kept as a dated baseline in [benchmarks/lamaria_r03.json](benchmarks/lamaria_r03.json).

These are same-recording reference checks, **not cross-recording accuracy** or proof that unmapped areas are rejected. Timings are end-to-end measurements of this implementation, not a controlled per-image comparison. The cells here remain unexecuted in source; these numbers are not fabricated notebook outputs.

R_03 has no recorded VIO. Do not replace VIO with closed-loop MPS or densified pseudo-ground truth. Use [pilot_replay.ipynb](pilot_replay.ipynb) for that separate Gen 2 workflow.

Keep raw VRS/MPS, maps/features, model weights and executed caches out of release source. LaMAria data/media attribution: Anusha Krishnan et al., ICCV 2025; [official dataset](https://lamaria.ethz.ch), [CC BY 4.0](https://github.com/cvg/lamaria/blob/main/LICENSE-CC-BY).